# 🤖 JARVIS Mobile — build your own voice assistant in Google Colab

Run the cells top to bottom. The last cell prints a **public link** — open it on your phone, tap the mic, and talk to JARVIS.

**The algorithm (one turn):**
1. **Listen** — phone mic records audio → `faster-whisper` converts speech to text (GPU)
2. **Gate** — optional wake word (“Jarvis …”) is detected and stripped
3. **Think** — Anthropic Claude API reasons over the request with short-term memory
4. **Act** — Claude calls skills via tool use: time, calculator, weather (Open-Meteo), web lookup, notes, reminders
5. **Speak** — reply converted to voice with `edge-tts` (fallback `gTTS`) and auto-played on the phone
6. **Remember** — conversation window + notes/reminders saved to JSON

**Before you start:** `Runtime → Change runtime type → T4 GPU`, and add your key under the 🔑 **Secrets** panel (left sidebar) as `ANTHROPIC_API_KEY` with *Notebook access* switched on.

## 1 · Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'No GPU — JARVIS will run on CPU (slower speech recognition)'

## 2 · Get the code and install dependencies (~1–2 min)

In [ ]:
!rm -rf jarvis-mobile && git clone -q https://github.com/skiran012/jarvis-mobile.git
%cd jarvis-mobile
!pip install -q -r requirements.txt
print("✅ Installed")

## 3 · Load your API key
If you saved `ANTHROPIC_API_KEY` in 🔑 Secrets it is used automatically; otherwise a box appears — paste the key (starts with `sk-ant-`) and press Enter.

In [ ]:
import os
from getpass import getpass

def get_secret(name, required=True):
    """Read from Colab Secrets (🔑); if missing, ask in a text box — easier on a phone."""
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    if required:
        return getpass(f"Paste your {name} and press Enter: ").strip()
    return ""

os.environ["ANTHROPIC_API_KEY"] = get_secret("ANTHROPIC_API_KEY")
os.environ["TAVILY_API_KEY"] = get_secret("TAVILY_API_KEY", required=False)  # optional
assert os.environ["ANTHROPIC_API_KEY"].startswith("sk-ant-"), "That doesn't look like an Anthropic key (it should start with sk-ant-)"
print("✅ Keys loaded")

## 4 · Personalise JARVIS (optional)

In [ ]:
os.environ["JARVIS_USER"] = "Sir"                     # how JARVIS addresses you
os.environ["JARVIS_TZ"] = "Asia/Kolkata"
os.environ["JARVIS_EDGE_VOICE"] = "en-GB-RyanNeural"    # try en-IN-PrabhatNeural, hi-IN-MadhurNeural, kn-IN-GaganNeural
os.environ["JARVIS_WHISPER_MODEL"] = "small"            # tiny/base = faster, medium/large-v3 = more accurate
os.environ["JARVIS_WAKE_WORD"] = "false"                # "true" = only respond when you start with "Jarvis"
os.environ["JARVIS_MODEL"] = "claude-sonnet-5-5"

## 5 · Smoke test (text only)

In [ ]:
from jarvis.pipeline import Jarvis

jarvis = Jarvis()
for q in ["What's the time?", "What's 18 percent of 2450?", "Weather in Bengaluru?"]:
    r = jarvis.turn(text=q, voice=False)
    print(f"🧑 {q}\n🤖 {r.reply}   [{', '.join(r.tools_used) or 'no tools'} · {r.latency_s}s]\n")

## 6 · Launch on your phone 📱
Run the cell below and wait ~30 seconds. A big blue **Open JARVIS** button appears under it — tap it (the link looks like `https://abc123.gradio.live`). Allow the microphone, then tap **Tap to talk**.

Keep this Colab tab open while you use JARVIS. Add to home screen: Android Chrome **⋮ → Add to Home screen**; iPhone Safari **Share → Add to Home Screen**.

In [ ]:
from IPython.display import HTML, display
from jarvis.app import build_app

app = build_app(jarvis)
_, local_url, share_url = app.queue().launch(share=True, prevent_thread_lock=True, quiet=True)

if not share_url:  # fallback: Colab's own proxy link (works in a browser signed in to this Google account)
    from google.colab.output import eval_js
    share_url = eval_js("google.colab.kernel.proxyPort(7860)")

display(HTML(f"""
<div style="text-align:center;padding:16px">
  <a href="{share_url}" target="_blank"
     style="display:inline-block;padding:18px 28px;font-size:22px;font-weight:700;color:#fff;
            background:#0b84ff;border-radius:14px;text-decoration:none">🤖 Open JARVIS</a>
  <p style="font-size:16px;word-break:break-all">{share_url}</p>
  <p style="font-size:13px;color:#888">Long-press the link to copy it. Valid while this notebook keeps running.</p>
</div>"""))

### Notes
- Keep this tab open: the phone link works only while the Colab runtime is running (free Colab disconnects after inactivity, and the share link expires after about a week).
- For a link that stays up permanently, deploy the same repo to **Hugging Face Spaces** (Gradio SDK) — see the README.
- Add a new skill: write a function in `jarvis/skills.py`, add its schema to `TOOLS`, register it in `REGISTRY`.